# 08 — Saved pipeline check

This notebook checks the pipeline file that the Streamlit app loads. It does not fit a model, and it does not rebuild the encoder. The file was written by `python -m app.export_frozen_pipeline` from the frozen Phase 6 settings. Notebooks 01 to 07 remain the record of the audit, the models, and the locked-test scores.


## 1. Setup


In [1]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
project_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "config.py").is_file():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError("Start JupyterLab from the project folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.predict import load_pipeline, pipeline_path, predict_request
from src import config
from src.preprocessing import MODEL_FEATURES

artifact_path = pipeline_path()
if not artifact_path.is_file():
    raise FileNotFoundError(
        "The saved pipeline is missing. Run python -m app.export_frozen_pipeline "
        "from the project folder. This notebook does not train a model."
    )
pipeline = load_pipeline(artifact_path)
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Pipeline file: {artifact_path}")
print(f"Format: {pipeline['format']}")
print(f"Features: {pipeline['features']}")
if pipeline["features"] != list(MODEL_FEATURES):
    raise RuntimeError("The saved features do not match the training feature list.")
if "Work_Center" in pipeline["features"]:
    raise RuntimeError("Work Center is in the saved feature list.")
print(f"Early stopping: {pipeline['model'].early_stopping}")
print(f"Iteration cap: {pipeline['max_iter']}")
print(f"Selected setting: {pipeline['selected']}")


Project root: /workspace
Pipeline file: /workspace/models/frozen_resolution_pipeline.joblib
Format: ida117v-frozen-hgb-v1
Features: ['C3_Complaint_Type', 'Suburb', 'Ward', 'Sub_Council', 'created_day_of_week', 'created_month', 'created_is_weekend']
Early stopping: False
Iteration cap: 12
Selected setting: untuned


## 2. Agreement with the locked-test file

The export compared this refit with the predictions saved by notebook 06. This cell reads that stored comparison. It does not score the locked test again and it does not change the model.


In [2]:
comparison = pipeline["locked_test_comparison"]
print(comparison)
if not comparison.get("compared"):
    raise RuntimeError("The export did not compare the refit with the saved locked-test predictions.")
if int(comparison["matching_predictions"]) != int(comparison["rows"]):
    raise RuntimeError("The saved pipeline does not reproduce the notebook 06 predictions.")
sample = {key: value for key, value in pipeline["sample_input"].items() if key != "description"}
result = predict_request(pipeline, sample)
print(pipeline["sample_input"]["description"])
print(result["prediction"])
print({label: round(score, 6) for label, score in result["probabilities"].items()})
check = {
    "pipeline_file": str(artifact_path.relative_to(config.PROJECT_ROOT)),
    "features": pipeline["features"],
    "selected": pipeline["selected"],
    "max_iter": int(pipeline["max_iter"]),
    "locked_test_rows_matched": int(comparison["matching_predictions"]),
    "locked_test_rows": int(comparison["rows"]),
    "demonstration_prediction": result["prediction"],
    "refit_performed_in_this_notebook": False,
}
path = config.TABLES_DIR / "08_export_check.json"
path.write_text(json.dumps(check, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {path}")


{'compared': True, 'rows': 422033, 'matching_predictions': 422033, 'max_probability_gap': 0.0}
Demonstration only. Each field is a frequent training category, skipping the stored placeholders '#' and 'Not assigned'. Those placeholders remain in the dropdowns. The fields are entered together so the form can be tried. This is not a logged municipal request and it is not an unresolved case.
Resolved Fast
{'Resolved Fast': 0.598355, 'Resolved Slow': 0.247213, 'Unresolved': 0.154433}
Wrote /workspace/results/tables/08_export_check.json
